# Experiment 1: Classification

## 1.1 importing data and examining data

In [57]:
                                           

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score


clf_df = pd.read_csv("classification_escalation.csv")

print("Shape:", clf_df.shape)
display(clf_df.head())
print("\nMissing values per column:\n", clf_df.isnull().sum())
print("\nTarget distribution (proportions):\n", clf_df["escalated"].value_counts(normalize=True))

Shape: (900, 7)


,ticket_id,complexity_score,team_experience_years,priority,channel,customer_tenure_months,escalated
0,T0000,2.0,15.0,Medium,Email,61.818896,0
1,T0001,9.0,6.0,Critical,Chat,110.024317,0
2,T0002,10.0,8.0,High,Phone,36.311244,0
3,T0003,2.0,1.0,Medium,Phone,85.127855,0
4,T0004,8.0,14.0,Medium,Chat,18.225059,0



Missing values per column:
 ticket_id                  0
complexity_score          22
team_experience_years     36
priority                  13
channel                    9
customer_tenure_months    31
escalated                  0
dtype: int64

Target distribution (proportions):
 escalated
0    0.787778
1    0.212222
Name: proportion, dtype: float64


## 1.2 Feature/Target Separation & Stratified Train/Test Split

In [58]:

X = clf_df.drop(columns=["ticket_id", "escalated"])
y = clf_df["escalated"]


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Train set: {X_train.shape[0]} rows | Escalated rate: {y_train.mean():.4f}")
print(f"Test set:  {X_test.shape[0]} rows | Escalated rate: {y_test.mean():.4f}")

Train set: 720 rows | Escalated rate: 0.2125
Test set:  180 rows | Escalated rate: 0.2111


## 1.3 Leakage-Safe Preprocessing Pipelin

In [59]:

num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X_train.select_dtypes(include=["str", "category"]).columns.tolist()

print("Numeric columns:", num_cols)
print("Categorical columns:", cat_cols)

num_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


cat_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])


preprocessor = ColumnTransformer(transformers=[
    ("num", num_transformer, num_cols),
    ("cat", cat_transformer, cat_cols)
])

Numeric columns: ['complexity_score', 'team_experience_years', 'customer_tenure_months']
Categorical columns: ['priority', 'channel']


##  1.4 Baseline Model Comparison (5-Fold Stratified CV)

In [60]:
# Identical 5-fold stratified CV splitter for fair comparison
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "k-NN": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(random_state=42)
}

cv_results = []

for name, model in models.items():
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", model)
    ])
    scores = cross_validate(
        pipe, X_train, y_train,
        cv=cv,
        scoring="f1",
        return_train_score=True
    )
    cv_results.append({
        "Model": name,
        "Mean CV F1": round(scores["test_score"].mean(), 4),
        "Std CV F1": round(scores["test_score"].std(), 4),
        "Mean Train F1": round(scores["train_score"].mean(), 4),
        "Fold F1 Scores": np.round(scores["test_score"], 3)
    })

results_df = pd.DataFrame(cv_results)
display(results_df)

,Model,Mean CV F1,Std CV F1,Mean Train F1,Fold F1 Scores
0,Logistic Regression,0.3927,0.0627,0.4400,"[0.391, 0.326, 0.364, 0.372, 0.511]"
1,k-NN,0.3608,0.0503,0.5244,"[0.367, 0.326, 0.426, 0.286, 0.4]"
2,Decision Tree,0.3307,0.0581,1.0000,"[0.4, 0.23, 0.323, 0.373, 0.328]"


## 1.5 Hyperparameter Tuning with GridSearchCV

In [61]:
pipe_lr = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])


param_grid = {
    "classifier__C": [0.01, 0.1, 1.0, 10.0, 100.0],
    "classifier__class_weight": [None, "balanced"]
}

grid_search = GridSearchCV(
    estimator=pipe_lr,
    param_grid=param_grid,
    cv=cv,
    scoring="f1",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

best_idx = grid_search.best_index_
best_mean = grid_search.best_score_
best_std = grid_search.cv_results_["std_test_score"][best_idx]

print("Best Parameters:", grid_search.best_params_)
print(f"Best Tuned CV F1: {best_mean:.4f} ± {best_std:.4f}")

Best Parameters: {'classifier__C': 0.1, 'classifier__class_weight': 'balanced'}
Best Tuned CV F1: 0.5200 ± 0.0423


## 1.6 Final One-Time Evaluation on the Held-Out Test Set

In [62]:
best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)

print(f"Final Test F1-Score: {f1_score(y_test, y_pred):.4f}\n")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Final Test F1-Score: 0.6000

Confusion Matrix:
 [[117  25]
 [ 11  27]]

Classification Report:
               precision    recall  f1-score   support

           0       0.91      0.82      0.87       142
           1       0.52      0.71      0.60        38

    accuracy                           0.80       180
   macro avg       0.72      0.77      0.73       180
weighted avg       0.83      0.80      0.81       180



# Experiment 2: Regression

## 1.1 Imports, Loading Data & Inspecting Outliers

In [63]:
                                      

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold, cross_validate, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

reg_df = pd.read_csv("resolution_time_regression.csv")

print("Shape:", reg_df.shape)
display(reg_df.head())
print("\nMissing values per column:\n", reg_df.isnull().sum())
print("\nTarget summary (resolution_time_hours):\n", reg_df["resolution_time_hours"].describe())

# Detect outliers in the target using the 1.5 * IQR rule
y_all = reg_df["resolution_time_hours"]
q1, q3 = y_all.quantile(0.25), y_all.quantile(0.75)
iqr = q3 - q1
upper_fence = q3 + 1.5 * iqr
outliers = y_all[y_all > upper_fence]

print(f"\nIQR Upper Fence: {upper_fence:.2f} hours")
print(f"Outliers found: {len(outliers)} ({len(outliers)/len(y_all)*100:.2f}% of data)")
print("Top 5 longest resolution times:", np.round(y_all.nlargest(5).values, 2))

Shape: (800, 7)


,ticket_id,complexity_score,team_experience_years,priority,channel,customer_tenure_months,resolution_time_hours
0,R0000,7.0,10.0,Medium,Chat,32.638377,8.38724
1,R0001,4.0,15.0,Medium,Chat,45.273760,0.50000
2,R0002,4.0,13.0,Low,Portal,44.878323,0.50000
3,R0003,1.0,10.0,High,Phone,40.095034,0.50000
4,R0004,3.0,15.0,Low,Portal,47.451134,0.50000



Missing values per column:
 ticket_id                  0
complexity_score          16
team_experience_years     24
priority                   0
channel                    0
customer_tenure_months    20
resolution_time_hours      0
dtype: int64

Target summary (resolution_time_hours):
 count    800.000000
mean       5.873160
std        4.144974
min        0.500000
25%        2.631319
50%        5.387386
75%        8.345903
max       27.488229
Name: resolution_time_hours, dtype: float64

IQR Upper Fence: 16.92 hours
Outliers found: 10 (1.25% of data)
Top 5 longest resolution times: [27.49 26.62 26.14 24.11 24.1 ]


## 1.2 Feature/Target Separation & Train/Test Split

In [64]:

X_reg = reg_df.drop(columns=["ticket_id", "resolution_time_hours"])
y_reg = reg_df["resolution_time_hours"]


X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

print(f"Train set: {X_train_r.shape[0]} rows | Mean target: {y_train_r.mean():.2f} hrs")
print(f"Test set:  {X_test_r.shape[0]} rows | Mean target: {y_test_r.mean():.2f} hrs")

Train set: 640 rows | Mean target: 5.77 hrs
Test set:  160 rows | Mean target: 6.30 hrs


## 1.3 Leakage-Safe Preprocessing Pipeline

In [65]:
num_cols_r = X_train_r.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols_r = X_train_r.select_dtypes(include=["str", "category"]).columns.tolist()

num_transformer_r = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_transformer_r = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_r = ColumnTransformer(transformers=[
    ("num", num_transformer_r, num_cols_r),
    ("cat", cat_transformer_r, cat_cols_r)
])

## 1.4 Baseline Model Comparison (5-Fold CV with MAE)

In [66]:

cv_reg = KFold(n_splits=5, shuffle=True, random_state=42)

reg_models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(random_state=42),
    "Decision Tree Regressor": DecisionTreeRegressor(random_state=42)
}

reg_results = []

for name, model in reg_models.items():
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor_r),
        ("regressor", model)
    ])
   
    scores = cross_validate(
        pipe, X_train_r, y_train_r,
        cv=cv_reg,
        scoring={
            "mae": "neg_mean_absolute_error",
            "rmse": "neg_root_mean_squared_error"
        },
        return_train_score=True
    )
    # Negate the scores back to positive hours
    val_mae = -scores["test_mae"]
    train_mae = -scores["train_mae"]
    val_rmse = -scores["test_rmse"]

    reg_results.append({
        "Model": name,
        "Mean CV MAE (hrs)": round(val_mae.mean(), 4),
        "Std CV MAE": round(val_mae.std(), 4),
        "Mean Train MAE": round(train_mae.mean(), 4),
        "Mean CV RMSE (hrs)": round(val_rmse.mean(), 4),
        "Fold MAEs": np.round(val_mae, 3)
    })

reg_results_df = pd.DataFrame(reg_results)
display(reg_results_df)

,Model,Mean CV MAE (hrs),Std CV MAE,Mean Train MAE,Mean CV RMSE (hrs),Fold MAEs
0,Linear Regression,2.0351,0.1717,1.9851,2.9606,"[2.342, 1.817, 1.965, 2.044, 2.008]"
1,Ridge Regression,2.0351,0.1714,1.9855,2.9605,"[2.341, 1.816, 1.967, 2.045, 2.006]"
2,Decision Tree Regressor,3.2391,0.0538,0.0000,4.5876,"[3.241, 3.168, 3.216, 3.334, 3.236]"


## 1.5 Hyperparameter Tuning (GridSearchCV for Ridge alpha and Tree max_depth)

In [67]:
#  Tune Ridge alpha
pipe_ridge = Pipeline(steps=[
    ("preprocessor", preprocessor_r),
    ("regressor", Ridge(random_state=42))
])
param_grid_ridge = {
    "regressor__alpha": [0.01, 0.1, 1.0, 5.0, 10.0, 25.0, 50.0, 100.0]
}
grid_ridge = GridSearchCV(
    pipe_ridge, param_grid_ridge,
    cv=cv_reg, scoring="neg_mean_absolute_error", n_jobs=-1
)
grid_ridge.fit(X_train_r, y_train_r)

ridge_idx = grid_ridge.best_index_
print("Best Ridge Params:", grid_ridge.best_params_)
print(f"Tuned Ridge CV MAE: {-grid_ridge.best_score_:.4f} ± {grid_ridge.cv_results_['std_test_score'][ridge_idx]:.4f}")

#  Tune Decision Tree max_depth
pipe_dt_r = Pipeline(steps=[
    ("preprocessor", preprocessor_r),
    ("regressor", DecisionTreeRegressor(random_state=42))
])
param_grid_dt_r = {
    "regressor__max_depth": [2, 3, 4, 5, 6, 8, 10, None]
}
grid_dt_r = GridSearchCV(
    pipe_dt_r, param_grid_dt_r,
    cv=cv_reg, scoring="neg_mean_absolute_error", n_jobs=-1
)
grid_dt_r.fit(X_train_r, y_train_r)

dt_idx = grid_dt_r.best_index_
print("\nBest Tree Params:", grid_dt_r.best_params_)
print(f"Tuned Tree CV MAE:  {-grid_dt_r.best_score_:.4f} ± {grid_dt_r.cv_results_['std_test_score'][dt_idx]:.4f}")

Best Ridge Params: {'regressor__alpha': 0.1}
Tuned Ridge CV MAE: 2.0351 ± 0.1716

Best Tree Params: {'regressor__max_depth': 3}
Tuned Tree CV MAE:  2.4540 ± 0.1825


## 1.6 Final One-Time Evaluation on the Held-Out Test Set

In [68]:
best_reg_model = grid_ridge.best_estimator_
y_pred_r = best_reg_model.predict(X_test_r)

test_mae = mean_absolute_error(y_test_r, y_pred_r)
test_rmse = root_mean_squared_error(y_test_r, y_pred_r)

print(f"Final Test MAE  (Tuned Ridge): {test_mae:.4f} hours")
print(f"Final Test RMSE (Tuned Ridge): {test_rmse:.4f} hours")

# Inspect the 5 largest prediction errors on the test set
error_df = pd.DataFrame({
    "Actual_Hours": y_test_r,
    "Predicted_Hours": np.round(y_pred_r, 2),
    "Absolute_Error": np.round(np.abs(y_test_r - y_pred_r), 2)
})
display(error_df.sort_values("Absolute_Error", ascending=False).head(5))

Final Test MAE  (Tuned Ridge): 2.2301 hours
Final Test RMSE (Tuned Ridge): 3.4028 hours


,Actual_Hours,Predicted_Hours,Absolute_Error
398,27.488229,6.38,21.11
49,24.108510,4.84,19.27
537,15.582888,3.36,12.22
490,5.110015,12.25,7.14
306,5.311855,10.81,5.50


# Lab 4 Report: Leakage-Safe Classification & Regression Pipelines

## 1. Problem Framing:

    This lab addresses two operational prediction tasks for a customer support ticketing system using historical ticket records (complexity_score,     team_experience_years, priority, channel, and customer_tenure_months), excluding the non-predictive identifier ticket_id in both tasks:

 ### - Experiment A — Escalation Classification (classification_escalation.csv, 900 tickets):
       A binary classification task predicting escalated (0 = resolved normally, 1 = escalated). The target is imbalanced (78.8% class 0, 21.2% class 1), making F1-score the primary evaluation metric rather than accuracy, as it balances precision and recall on the minority escalation class.

### - Experiment B — Resolution Time Regression (resolution_time_regression.csv, 800 tickets): 
     A continuous regression task predicting resolution_time_hours. Ticket resolution times are right-skewed (median 5.39 hours, IQR 2.63–8.35 hours)      with 10 extreme outliers (1.25% of tickets exceeding the 16.92-hour IQR upper fence, up to 27.49 hours). Mean Absolute Error (MAE) is used as the      primary metric because it measures average error directly in hours and is less distorted by extreme tail outliers than RMSE.


## 2. Preprocessing & Leakage Prevention:
   
     Both datasets contain missing values across numeric features (and across priority and channel in Experiment A). To prepare the features without  introducing data leakage, all transformations are encapsulated inside a ColumnTransformer within a scikit-learn Pipeline: 
 ### - Numeric Pipeline (complexity_score, team_experience_years, customer_tenure_months):
      - SimpleImputer(strategy="median"): Replaces missing numeric values with the training-fold median, which is robust against skewed distributions   and outliers.

      - StandardScaler(): Standardizes features to zero mean and unit variance. This is essential for distance-based models (k-NN) and L2-regularized linear models (Logistic Regression and Ridge Regression) so that large-scale features like customer_tenure_months (up to 120) do not dominate smaller-scale features like complexity_score (1–10).

  ### - Categorical Pipeline (priority, channel):

       -SimpleImputer(strategy="most_frequent"): Fills missing categories using the training-fold mode.

       -OneHotEncoder(handle_unknown="ignore"): Converts nominal categories into binary indicator columns while safely ignoring unseen categories at inference time.

     
     -Leakage Prevention; Because the ColumnTransformer sits inside the Pipeline, .fit_transform() is executed strictly on the training folds during each cross-validation split, while .transform() applies those learned medians, means, standard deviations, and encodings to the validation fold and later to the held-out test set. No statistics from validation or test data ever influence model training.

## 3. Validation Strategy

   ### -Train/Test Holdout Split (80/20, random_state=42):
   
        - Experiment A: Uses a stratified split (stratify=y), preserving the original class imbalance across both sets (Train: 720 rows, 21.25% escalated; Test: 180 rows, 21.11% escalated).
   
        - Experiment B: Uses a standard random split (Train: 640 rows; Test: 160 rows) since the target is continuous.
   
        -In both experiments, the test set is locked away until model selection and hyperparameter tuning are complete, and evaluated only once
   
### - Cross-Validation (5-Fold):
     
     - Both experiments  use identical StratifiedKFold(n_splits=5, shuffle=True, random_state=42) splits across all models, ensuring every fold maintains the ~79/21 class ratio and models are compared on the exact same folds.

## 4. Candidate Models & Hyperparameter Search Space

  ### - Experiment A (Classification)
    
     - Candidates: Logistic Regression (linear baseline), k-NN (non-parametric distance-based model), and Decision Tree (non-linear rule-based model).
    
      -Search Space (Tuned Candidate — Logistic Regression via GridSearchCV):

          classifier__C: [0.01, 0.1, 1.0, 10.0, 100.0] — controls inverse L2 regularization strength to balance bias and variance.

         classifier__class_weight: [None, "balanced"] — tests whether inversely weighting the minority escalation class improves F1-score under the 79/21 imbalance.
    
  ### - Experiment B (Regression)
    
          - Candidates: Linear Regression (unregularized linear baseline), Ridge Regression (L2-regularized linear model), and Decision Tree Regressor (non-linear partition model).
    
           - Search Space (GridSearchCV optimizing neg_mean_absolute_error):
    
                    Ridge Regression: regressor__alpha: [0.01, 0.1, 1.0, 5.0, 10.0, 25.0, 50.0, 100.0] — controls L2 shrinkage strength.

                  Decision Tree Regressor: regressor__max_depth: [2, 3, 4, 5, 6, 8, 10, None] — constrains tree depth to curb the severe overfitting seen in the default tree.

## 5. Final Test Set Results & Error Analysis

   ### - Experiment A: Tuned Logistic Regression (C=0.1, class_weight="balanced")
        - Test F1-Score (Class 1): 0.6000 (Precision: 0.52, Recall: 0.71, Accuracy: 0.80).
        -  Confusion Matrix (180 test tickets):
          
            - True Negatives (117): Routine tickets correctly left unescalated.

            - True Positives (27): Escalated tickets correctly flagged early.

            - False Positives (25): Routine tickets unnecessarily flagged for escalation—creating extra review workload and potential alert fatigue for senior support staff.

            - False Negatives (11): Escalated tickets missed by the model—risking SLA breaches, longer resolution delays, and customer dissatisfaction.
         
        - Because class_weight="balanced" penalizes missed escalations heavily, the model trades higher False Positives (25) to keep False Negatives low (11, capturing 71% of actual escalations).
          
###  - Experiment B: Tuned Ridge Regression (alpha=0.1)
        - Final Test MAE: 2.2301 hours (consistent with the CV MAE of 2.0351 ± 0.1716 hours, confirming good generalization without leakage).
      
        - Final Test RMSE: 3.4028 hours.

        - Outliers & MAE vs. RMSE: The 1.17-hour gap between Test MAE (2.23 hrs) and Test RMSE (3.40 hrs) is driven by extreme outlier tickets in the test set—specifically two tickets that took 27.49 hours and 24.11 hours while the model predicted 6.38 hours and 4.84 hours (absolute errors of 21.11 and 19.27 hours). Because MAE penalizes errors linearly, it favors models that predict typical tickets accurately (optimizing toward the conditional median). Because RMSE squares errors before averaging, a 20-hour miss is penalized 100× more than a 2-hour miss, meaning RMSE can prefer a different model that sacrifices accuracy on normal tickets just to pull predictions closer to rare extreme outliers.
     
## 7. Final Model Justification & Limitations
 ###  Experiment A :
       Tuned Logistic Regression (C=0.1, class_weight="balanced"): Chosen because it achieved the highest CV F1 (0.5200) and lowest fold standard deviation (0.0423), generalized strongly to the test set (0.6000 F1), runs with negligible inference latency, and provides interpretable feature coefficients.

       Limitations: Linear decision boundaries cannot capture complex feature interactions ,for instance, high complexity combined with low team experience, and a 0.52 precision means nearly half of escalation alerts are false alarms.

### Experiment B — 

    Tuned Ridge Regression (alpha=0.1): Chosen because it outperformed both unregularized Linear Regression and the tuned Decision Tree on both CV MAE (2.0351 hrs) and CV RMSE (2.9606 hrs) with stable fold variance.

        Limitations: Ridge cannot predict rare, extreme resolution delays (18–27+ hours) because those spikes are likely caused by unmeasured external factors like; customer responsiveness, vendor outages missing from the current feature set.
   

# Refelction questions:

## 1. Why stratify the classification split?
   
With an imbalanced 78.8% / 21.2% target distribution, a purely random split could accidentally produce train, validation, or test sets with skewed escalation rates, making F1 estimates noisy and unreliable. Stratifying guarantees that every split preserves the exact ~21.2% minority-class proportion for consistent training and fair evaluation.

## 2. Why must preprocessing be inside the pipeline?

Fitting imputers, scalers, or encoders before splitting leaks validation/test distribution statistics (medians, means, standard deviations) into training, artificially inflating CV scores. Wrapping preprocessing inside a Pipeline ensures statistics are computed exclusively on each training fold and applied cleanly to unseen validation and test folds.

## 3. What does fold variability tell you?

   
Standard deviation across CV folds measures how sensitive a model is to which subset of data it is trained and evaluated on. High fold variability signals either small minority-class counts per fold (as in Experiment A, where tuning lowered std from 0.0627 to 0.0423) or uneven distribution of extreme outliers across folds (as in Experiment B's RMSE std of 0.5844 hrs).

## 4. Why should the test set be evaluated only once?

   
The test set serves as an unbiased proxy for unseen production data. Iterating on model choices or hyperparameters after seeing test results turns the test set into a second validation set, leaking information and producing overly optimistic performance claims that will not hold in deployment.

## 5. What deployment constraints might change your choice?
   
Production constraints such as inference latency and memory (favoring Logistic/Ridge over memory-heavy k-NN at scale), stakeholder interpretability (favoring linear weights or shallow trees over black-box models), asymmetric business costs (adjusting classification thresholds if Tier-2 staff cannot absorb 25 false positives, or if SLA breach penalties make false negatives unacceptable), and ongoing data drift (requiring safe unseen-category handling and cheap periodic retraining) can all override raw CV scores when selecting a production model.


